# Exploratory Data Analysis with Python

**Python in Practice series · Dartmouth Research Computing**

In this workshop you'll explore a real research dataset and learn how to:

1. Read data files into Python
2. Inspect the structure of your data
3. Identify and transform variable types
4. Summarize data numerically
5. Explore basic relationships in your data

### The data

We'll use the **Palmer Penguins** dataset: size measurements for 344 penguins of three species,
observed on three islands in the Palmer Archipelago, Antarctica, from 2007 to 2009.

Data collected by Dr. Kristen Gorman and the Palmer Station LTER. Packaged by Horst, Hill & Gorman (2020),
*palmerpenguins: Palmer Archipelago (Antarctica) penguin data*. https://allisonhorst.github.io/palmerpenguins/

## 1. Setup

Import the libraries we'll use:
- `pandas` (as `pd`) for working with tables of data
- `seaborn` (as `sns`) for quick plots

In [ ]:
import pandas as pd
import seaborn as sns

## 2. Read in the data

Read `data/penguins.csv` into a DataFrame called `penguins` using `pd.read_csv()`, then look at the first few rows with `.head()`.

In [ ]:
penguins = pd.read_csv("data/penguins.csv")
penguins.head()

### Other file types

pandas has a reader for most common formats, and they all work the same way: give it a file path, get back a DataFrame.

| Format | Function |
|---|---|
| CSV | `pd.read_csv()` |
| Excel (.xlsx) | `pd.read_excel()` (useful options: `sheet_name=`, `skiprows=`) |
| Stata (.dta) | `pd.read_stata()` |
| SPSS (.sav) | `pd.read_spss()` |
| SAS | `pd.read_sas()` |
| Parquet | `pd.read_parquet()` |

In [ ]:
# Example only: these files aren't in the repo
# pd.read_excel("my_data.xlsx", sheet_name="Sheet1")
# pd.read_stata("my_data.dta")

## 3. Inspect the structure

How big is the dataset? Try `.shape` and `.columns`.

In [ ]:
penguins.shape

`.shape` gives (rows, columns). Note there are no parentheses: it's an attribute, not a method.

In [ ]:
penguins.columns

Get an overview of every column at once with `.info()`.

In [ ]:
penguins.info()

`.info()` is the most useful first look: number of rows, each column's type, and how many non-missing values each column has.

Look at a random handful of rows with `.sample()`. Why might this be better than `.head()`?

In [ ]:
penguins.sample(5)

`.head()` only shows the top of the file, which is often sorted in some way (here, all Adelie penguins come first). A random sample gives a more representative look.

How many distinct values does each column have? Try `.nunique()`.

In [ ]:
penguins.nunique()

Columns with only a few unique values (`species`, `island`, `sex`, `year`) are likely categorical, even if they're stored as numbers.

### Missing data

Count the missing values in each column with `.isna().sum()`.

In [ ]:
penguins.isna().sum()

Which rows are missing `bill_length_mm`? Use the missing-value check as a filter.

In [ ]:
penguins[penguins["bill_length_mm"].isna()]

Two penguins are missing nearly *every* measurement, while others are missing only `sex`. Those are different situations, and you might handle them differently in an analysis.

**🐧 Your turn:** Which rows are missing `sex`? How many are there?

In [ ]:
penguins[penguins["sex"].isna()]

## 4. Identify and transform variable types

Check the type of every column with `.dtypes`.

In [ ]:
penguins.dtypes

**Question:** flipper lengths are recorded as whole numbers. Why is `flipper_length_mm` a `float64`?

In [ ]:
penguins["flipper_length_mm"].head()

Missing values (`NaN`) are technically floats, so a single missing value turns a whole column of integers into floats. pandas has a special *nullable* integer type, `"Int64"` (capital I), that can hold missing values.

Convert `flipper_length_mm` to the nullable integer type `"Int64"` with `.astype()`.

In [ ]:
penguins["flipper_length_mm"] = penguins["flipper_length_mm"].astype("Int64")
penguins["flipper_length_mm"].head()

**🐧 Your turn:** Convert `body_mass_g` to `"Int64"` too.

In [ ]:
penguins["body_mass_g"] = penguins["body_mass_g"].astype("Int64")

### Text columns → categories

`species`, `island`, and `sex` are text columns with only a few possible values. Convert them to the `"category"` type.

In [ ]:
for col in ["species", "island", "sex"]:
    penguins[col] = penguins[col].astype("category")

What are the categories of `species`? Try `.cat.categories`.

In [ ]:
penguins["species"].cat.categories

**Question:** is `year` a number or a category? What would the *mean* year tell you?

In [ ]:
penguins["year"].mean()

The mean year (about 2008) isn't meaningful here. Year identifies *which* study season a penguin was measured in, so we'll treat it as a category.

Convert `year` to a category and check `.dtypes` again.

In [ ]:
penguins["year"] = penguins["year"].astype("category")
penguins.dtypes

## 5. Summarize data numerically

Get summary statistics for all numeric columns with `.describe()`.

In [ ]:
penguins.describe()

Check the min and max of each column: are they plausible? This is the fastest way to spot data entry errors or missing-value codes like `-99`.

Now summarize the categorical columns with `.describe(include="category")`.

In [ ]:
penguins.describe(include="category")

For categories, `top` is the most common value (the mode) and `freq` is how often it appears.

Calculate the mean, median, and mode of `body_mass_g`.

In [ ]:
penguins["body_mass_g"].mean()

In [ ]:
penguins["body_mass_g"].median()

In [ ]:
penguins["body_mass_g"].mode()

The mean is larger than the median, which suggests the distribution is slightly right-skewed. `.mode()` returns a Series rather than a single number because there can be ties.

### Counts and proportions

Count the penguins of each species with `.value_counts()`. Then add `normalize=True` to get proportions.

In [ ]:
penguins["species"].value_counts()

In [ ]:
penguins["species"].value_counts(normalize=True)

### Summaries by group

Calculate the mean, median, and standard deviation of `body_mass_g` for each species using `.groupby()` and `.agg()`.

In [ ]:
penguins.groupby("species", observed=True)["body_mass_g"].agg(["mean", "median", "std"])

`observed=True` tells pandas to show only categories that actually appear in the data (and avoids a warning).

**🐧 Your turn:**
1. Which island has the most penguins?
2. Which species has the longest median flipper length?

In [ ]:
penguins["island"].value_counts()

In [ ]:
penguins.groupby("species", observed=True)["flipper_length_mm"].median()

## 6. Explore basic relationships

### Two categorical variables

Which species live on which islands? Make a table of counts with `pd.crosstab()`.

In [ ]:
pd.crosstab(penguins["species"], penguins["island"])

Adelie penguins appear on all three islands, but Gentoo penguins appear only on Biscoe and Chinstrap penguins only on Dream.

### Two numeric variables

Calculate the correlation between every pair of numeric columns with `.corr(numeric_only=True)`.

In [ ]:
penguins.corr(numeric_only=True)

Notice that `year` isn't in this table, because we converted it to a category.

Plot `flipper_length_mm` against `body_mass_g` with `sns.scatterplot()`.

In [ ]:
sns.scatterplot(data=penguins, x="flipper_length_mm", y="body_mass_g")

A strong positive relationship: penguins with longer flippers tend to be heavier.

### A puzzle: bill length and bill depth

What's the correlation between `bill_length_mm` and `bill_depth_mm`? Plot them too.

In [ ]:
penguins[["bill_length_mm", "bill_depth_mm"]].corr()

In [ ]:
sns.scatterplot(data=penguins, x="bill_length_mm", y="bill_depth_mm")

Overall, the relationship is weakly negative: longer bills seem to be slightly shallower. But is that the whole story?

Make the same plot, but color the points by species with `hue="species"`.

In [ ]:
sns.scatterplot(data=penguins, x="bill_length_mm", y="bill_depth_mm", hue="species")

Now calculate the correlation *within* each species.

In [ ]:
penguins.groupby("species", observed=True)[["bill_length_mm", "bill_depth_mm"]].corr()

Within every species, the correlation is **positive**, even though it's negative overall. This is an example of **Simpson's paradox**: a relationship in combined data can reverse when you look within groups. Always explore your relationships by group.

**🐧 Your turn:** Compare body mass across species with a boxplot, using `sns.boxplot()`.

In [ ]:
sns.boxplot(data=penguins, x="species", y="body_mass_g")

## 7. Wrap-up: an EDA checklist

When you open a new dataset, ask:

1. **Shape:** How many rows and columns? Is that what you expected? (`.shape`, `.info()`)
2. **Types:** Is each column stored as the right type? (`.dtypes`, `.astype()`)
3. **Missingness:** What's missing, and is it missing in a pattern? (`.isna().sum()`)
4. **Ranges:** Are the min and max values plausible? (`.describe()`)
5. **Distributions:** What are the typical values and the spread? (`.mean()`, `.median()`, `.value_counts()`)
6. **Groups:** Do summaries differ across groups? (`.groupby()`)
7. **Relationships:** How do variables relate, overall and within groups? (`pd.crosstab()`, `.corr()`, scatterplots)

### Want more practice?

Try these on your own:
- Does the relationship between flipper length and body mass hold within each species?
- Do male and female penguins differ in body mass? Does that difference depend on species?
- Did the number of penguins measured change from year to year?